# 🎙️ Dynode — Voice Clone Notebook (Hindi, XTTS-v2)

**For campaign operators.** This trains a voice model from a speaker's audio and
gives you a **`cloned_voice_model.zip`** to import into the app. Once imported, the
voice appears in the app's dropdown next to Madhur & Swara and **speaks any name** —
no GPU needed on your PC.

**First:** *Runtime ▸ Change runtime type ▸ Hardware accelerator = T4 GPU ▸ Save.*
Then run each cell top-to-bottom (or Runtime ▸ Run all).

**For ~90% accuracy, upload 5–10 minutes of clean single-speaker speech.**
A short clip still trains, but similarity will be lower.


## 1 · Setup
_Ignore red pip warnings — normal on Colab. If a later cell fails on import, do Runtime ▸ Restart session and run again from here._


In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'   # accept the XTTS model license non-interactively

import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('⚠️  No GPU! Set Runtime ▸ Change runtime type ▸ T4 GPU, then Runtime ▸ Restart and rerun.')

!pip -q install coqui-tts faster-whisper 2>/dev/null
print('✅ install step done')

## 2 · Upload the speaker's audio
Upload a **video or audio** of the speaker. Longer + cleaner = better. 5–10 min of clear speech (no background music) is ideal.


In [ ]:
from google.colab import files
import wave, contextlib

print('Select the speaker video/audio file...')
up = files.upload()
src_path = list(up.keys())[0]
print('Got:', src_path)

REF_WAV = 'speaker_ref.wav'
!ffmpeg -y -i "$src_path" -vn -ac 1 -ar 22050 -af "highpass=f=70,afftdn=nf=-25,loudnorm=I=-16:TP=-1.5:LRA=11" "$REF_WAV" -hide_banner -loglevel error

with contextlib.closing(wave.open(REF_WAV,'r')) as w:
    dur = w.getnframes()/float(w.getframerate())
print(f'Clean reference: {REF_WAV}  ({dur:.1f} s)')
if dur < 240:
    print('⚠️  Under ~4 min — the model will still train but similarity may be lower. More clean speech = better.')

## 3 · Settings
Edit if you like, then run.


In [ ]:
LANGUAGE = 'hi'                       # Hindi
GREETING_TEMPLATE = '{} जी नमस्कार'   # used only for the sample below; the APP speaks real names
EPOCHS = 6                            # 6 is a good default on a T4
SAMPLE_NAME = 'राहुल'                  # name spoken in the quality-check sample
print('Sample will say →', GREETING_TEMPLATE.format(SAMPLE_NAME))

## 4 · Prepare the data (Whisper)
Transcribes and segments your audio into a small training set. A few minutes.


In [ ]:
from TTS.demos.xtts_ft_demo.utils.formatter import format_audio_list
print('Analysing audio with Whisper...')
train_csv, eval_csv, total_secs = format_audio_list([REF_WAV], target_language=LANGUAGE, out_path='dataset/')
print(f'Usable speech: {total_secs/60.0:.1f} min   train={train_csv}')

## 5 · Fine-tune the voice
On a T4 this is roughly 15–40 min depending on audio length and epochs. Keep the tab open.

_This is the most version-sensitive cell. If it errors, copy the message — we can adjust one argument._


In [ ]:
from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt
import glob, os

print('Training… (do not close the tab)')
out = train_gpt(
    custom_model='', version='main', language=LANGUAGE,
    num_epochs=EPOCHS, batch_size=3, grad_acumm=84,
    train_csv=train_csv, eval_csv=eval_csv, output_path='run/',
)
print('train_gpt returned:', out)

def _find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    return hits[-1] if hits else None

FT_CONFIG = _find('run/**/config.json')
FT_VOCAB  = _find('run/**/vocab.json')
FT_CKPT   = _find('run/**/best_model.pth') or _find('run/**/model.pth')
print('config :', FT_CONFIG)
print('vocab  :', FT_VOCAB)
print('ckpt   :', FT_CKPT)
assert FT_CONFIG and FT_VOCAB and FT_CKPT, 'Could not locate fine-tuned files; check the log above.'

## 6 · Listen to a sample
Make sure it sounds like the speaker before downloading the model.


In [ ]:
import torch, soundfile as sf, numpy as np
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

cfg = XttsConfig(); cfg.load_json(FT_CONFIG)
model = Xtts.init_from_config(cfg)
model.load_checkpoint(cfg, checkpoint_path=FT_CKPT, vocab_path=FT_VOCAB, use_deepspeed=False)
model.cuda()
gpt_lat, spk = model.get_conditioning_latents(audio_path=[REF_WAV], gpt_cond_len=30, max_ref_length=60)

o = model.inference(GREETING_TEMPLATE.format(SAMPLE_NAME), LANGUAGE, gpt_lat, spk,
                    temperature=0.65, repetition_penalty=5.0, top_p=0.8)
sf.write('sample.wav', np.asarray(o['wav']), 24000)
from IPython.display import Audio, display
print('Sample:'); display(Audio('sample.wav'))

## 7 · Package & download the model
Bundles the trained voice into **cloned_voice_model.zip**. It's large (~1–2 GB), so the download can take a while.

Then in the app: **Clone Voice ▸ Import Cloned Voice** and pick this file.


In [ ]:
import shutil, os, zipfile

pkg = 'cloned_voice_model'
if os.path.isdir(pkg): shutil.rmtree(pkg)
os.makedirs(pkg)
shutil.copy(FT_CONFIG, os.path.join(pkg, 'config.json'))
shutil.copy(FT_VOCAB,  os.path.join(pkg, 'vocab.json'))
shutil.copy(FT_CKPT,   os.path.join(pkg, 'model.pth'))
shutil.copy(REF_WAV,   os.path.join(pkg, 'speaker_ref.wav'))

zip_path = 'cloned_voice_model.zip'
# ZIP_STORED: model weights don't compress, so skip compression for speed.
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:
    for f in os.listdir(pkg):
        z.write(os.path.join(pkg, f), f)
size_gb = os.path.getsize(zip_path)/1e9
print(f'✅ Built {zip_path}  ({size_gb:.2f} GB). Downloading…')
files.download(zip_path)
print('If the download stalls, mount Google Drive and copy it there instead:')
print('  from google.colab import drive; drive.mount("/content/drive")')
print('  shutil.copy(zip_path, "/content/drive/MyDrive/")')